# Fine-tuning Qwen2.5-0.5B on Banking77 (LoRA)

Runtime > Change runtime type > T4 GPU. The free tier is enough. About 30 to 40 minutes end to end.

What happens here:

1. Load Banking77: 13k real customer banking messages sorted into 77 intent labels.
2. Score the base model zero-shot, with all 77 labels pasted into the prompt.
3. Fine-tune with LoRA, training without the label list in the prompt.
4. Score again.
5. Look at which intents get confused with which.

## Why the baseline gets the label list and the fine-tuned model doesn't

The baseline needs those 77 labels in its prompt or it has no idea what the classes are, and that
costs roughly 700 tokens per request. The fine-tuned model learns the label space from the weights,
so it runs on about 30.

That isn't a rigged comparison. It's the actual case for fine-tuning: pay a one-off training cost,
then get shorter prompts and cheaper inference from then on. Expect to be asked about it.

## 0. Setup

In [ ]:
!pip -q install -U "transformers>=4.44" "datasets>=2.20" "peft>=0.12" "accelerate>=0.33"

# Colab preinstalls torchao 0.10.0, and peft's is_torchao_available() raises an
# outright ImportError on anything below 0.16 - it fires the moment you call
# get_peft_model(), even though we never touch torchao. We aren't quantizing, so
# remove it rather than fight the torch/torchao version matrix.
!pip -q uninstall -y torchao

In [ ]:
import torch, transformers, datasets, peft
print("torch       ", torch.__version__)
print("transformers", transformers.__version__)
print("datasets    ", datasets.__version__)
print("peft        ", peft.__version__)

assert torch.cuda.is_available(), "No GPU. Runtime -> Change runtime type -> T4 GPU."
print("GPU         ", torch.cuda.get_device_name(0))

# Pick fp16 vs bf16 by compute capability. torch.cuda.is_bf16_supported() returns
# True on a T4 (sm_75) because it counts *emulated* bf16 - which is very slow.
# Real bf16 hardware starts at Ampere (sm_80).
CC    = torch.cuda.get_device_capability()
BF16  = CC[0] >= 8
DTYPE = torch.bfloat16 if BF16 else torch.float16
print("capability  ", f"sm_{CC[0]}{CC[1]}")
print("dtype       ", DTYPE, "(bf16 needs sm_80+)")

In [ ]:
# --- knobs ---------------------------------------------------------------
MODEL_ID    = "Qwen/Qwen2.5-0.5B-Instruct"
N_TRAIN     = None   # None = all ~10k. Set to e.g. 3000 for a fast first run.
N_EVAL      = 300    # held-out test examples scored. 300 gives +/- ~3% noise.
EPOCHS      = 2
LR          = 2e-4
MAXLEN      = 128
SEED        = 42
OUT_DIR     = "banking77-qwen0.5b-lora"
# -------------------------------------------------------------------------
import random, numpy as np
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

## 1. Data

Check the dataset loads and looks the way you expect before spending GPU time on it.

In [ ]:
from datasets import load_dataset, ClassLabel

# datasets 5.x removed support for dataset loading scripts, and PolyAI/banking77
# still ships a banking77.py. The Hub auto-converts every dataset to parquet on a
# refs/convert/parquet branch, so read that instead. Try in order, report which won.
def load_banking77():
    attempts = [
        ("parquet branch", lambda: load_dataset("PolyAI/banking77",
                                                revision="refs/convert/parquet")),
        ("default",        lambda: load_dataset("PolyAI/banking77")),
        ("mteb mirror",    lambda: load_dataset("mteb/banking77")),
    ]
    for name, fn in attempts:
        try:
            d = fn()
            print(f"loaded via: {name}")
            return d
        except Exception as e:
            print(f"  [skip] {name}: {str(e)[:110]}")
    raise RuntimeError("Could not load Banking77 from any source.")

raw = load_banking77()

# Normalise to: a "text" column, and a "label" column that is a ClassLabel whose
# .names are the human-readable intent strings. Each mirror stores this slightly
# differently, so handle all three shapes.
cols = raw["train"].column_names
feat = raw["train"].features["label"]

if isinstance(feat, ClassLabel):
    LABELS = feat.names                       # already ideal
elif "label_text" in cols:
    # mteb-style: label is an int id, label_text holds the string. Build the
    # id -> string mapping, then re-type the column.
    id2name = {}
    for r in raw["train"]:
        id2name.setdefault(r["label"], r["label_text"])
        if len(id2name) == 77:
            break
    LABELS = [id2name[i] for i in range(len(id2name))]
    raw = raw.cast_column("label", ClassLabel(names=LABELS))
else:
    # label column holds the strings directly
    LABELS = sorted(set(raw["train"]["label"]))
    raw = raw.cast_column("label", ClassLabel(names=LABELS))

# Guardrails. The previous run passed a len() check while LABELS was [0,1,2,...],
# which only blew up 4 cells later - so assert the type, not just the count.
assert len(LABELS) == 77, f"expected 77 intents, got {len(LABELS)}"
assert all(isinstance(l, str) for l in LABELS), f"labels are not strings: {LABELS[:5]}"
assert "text" in cols, f"no text column, got {cols}"

print(raw)
print(f"\n{len(LABELS)} labels, e.g.: {LABELS[:8]}")
print("\nSample rows:")
for r in raw["train"].select(range(5)):
    print(f"  {LABELS[r['label']]:<35} | {r['text']}")

In [ ]:
# Class balance - worth a glance. Banking77 is roughly balanced, which is convenient
# and also why plain accuracy is a defensible metric here.
from collections import Counter
c = Counter(raw["train"]["label"])
print("examples per class: min %d, max %d, mean %.1f" % (
    min(c.values()), max(c.values()), sum(c.values())/len(c)))

In [ ]:
train_ds = raw["train"].shuffle(seed=SEED)
if N_TRAIN:
    train_ds = train_ds.select(range(N_TRAIN))

test_ds    = raw["test"].shuffle(seed=SEED).select(range(N_EVAL))
test_texts = test_ds["text"]
test_gold  = [LABELS[i] for i in test_ds["label"]]
print(f"train {len(train_ds)}   eval {len(test_ds)}")

## 2. Model + tokenizer

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tok = AutoTokenizer.from_pretrained(MODEL_ID)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

# transformers v5 renamed the `torch_dtype` argument to `dtype`. On v4 the new
# name is silently swallowed and you end up in fp32, so branch explicitly
# rather than relying on a try/except.
V5 = int(transformers.__version__.split(".")[0]) >= 5
dtype_kw = {"dtype": DTYPE} if V5 else {"torch_dtype": DTYPE}
print("transformers major:", transformers.__version__.split(".")[0], "->", list(dtype_kw)[0])

model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **dtype_kw).to("cuda")
print(f"{model.num_parameters()/1e6:.0f}M params | dtype {next(model.parameters()).dtype}")

## 3. Prompting and scoring

Two prompt builders. One includes the label list, for the baseline. One doesn't, for the fine-tuned
model.

The scorer maps whatever the model emits back onto a valid label: exact match first, then a fuzzy
fallback for near misses like `card arrival` against `card_arrival`. It also counts how often that
fallback has to fire. A base model needs it constantly and a fine-tuned one barely at all, which is
a result in its own right. Fine-tuning buys format reliability, not only accuracy.

In [ ]:
import re, difflib
from tqdm.auto import tqdm

SYS_SHORT = "You are an intent classifier for a banking app. Reply with exactly one intent label and nothing else."
SYS_LONG  = (SYS_SHORT.replace("one intent label", "one intent label from this list")
             + "\n\n" + "\n".join(LABELS))

def build_prompt(text, with_labels):
    return tok.apply_chat_template(
        [{"role": "system", "content": SYS_LONG if with_labels else SYS_SHORT},
         {"role": "user",   "content": text}],
        tokenize=False, add_generation_prompt=True)

print("prompt tokens  with labels:",
      len(tok(build_prompt("my card is broken", True))["input_ids"]))
print("prompt tokens without     :",
      len(tok(build_prompt("my card is broken", False))["input_ids"]))

In [ ]:
def _norm(s):
    s = s.strip().lower().split("\n")[0]
    s = re.sub(r"[^a-z0-9_ ]", "", s)
    return re.sub(r"[ _]+", "_", s).strip("_")

LOOKUP = {_norm(l): l for l in LABELS}

def to_label(raw_text):
    # -> (label_or_None, was_exact)
    n = _norm(raw_text)
    if n in LOOKUP:
        return LOOKUP[n], True
    m = difflib.get_close_matches(n, list(LOOKUP), n=1, cutoff=0.6)
    return (LOOKUP[m[0]], False) if m else (None, False)

@torch.no_grad()
def predict(m, texts, with_labels, bs=16, max_new_tokens=12):
    tok.padding_side = "left"          # left-pad for generation
    m.eval()
    raws = []
    for i in tqdm(range(0, len(texts), bs), desc="generating"):
        chunk = [build_prompt(t, with_labels) for t in texts[i:i + bs]]
        enc = tok(chunk, return_tensors="pt", padding=True,
                  add_special_tokens=False).to(m.device)
        gen = m.generate(**enc, max_new_tokens=max_new_tokens,
                         do_sample=False, pad_token_id=tok.pad_token_id)
        for j in range(len(chunk)):
            raws.append(tok.decode(gen[j][enc["input_ids"].shape[1]:],
                                   skip_special_tokens=True).strip())
    return raws

def score(raws, gold, tag):
    preds, exact = zip(*[to_label(r) for r in raws])
    acc      = sum(p == g for p, g in zip(preds, gold)) / len(gold)
    clean    = sum(exact) / len(gold)
    unparsed = sum(p is None for p in preds) / len(gold)
    print(f"\n=== {tag} ===")
    print(f"  accuracy              {acc:.1%}")
    print(f"  emitted a valid label {clean:.1%}   (needed fuzzy repair otherwise)")
    print(f"  unparseable           {unparsed:.1%}")
    return {"tag": tag, "accuracy": acc, "clean_format": clean,
            "unparseable": unparsed, "preds": list(preds)}

## 4. Baseline

A 0.5B model choosing among 77 fine-grained options is out of its depth, and it tends to latch onto
a handful of familiar-sounding labels. Read the raw outputs below, not only the number.

In [ ]:
base_raw = predict(model, test_texts, with_labels=True)
base = score(base_raw, test_gold, "BASELINE (zero-shot, labels in prompt)")

print("\nRaw outputs, first 10:")
for r, g in list(zip(base_raw, test_gold))[:10]:
    print(f"  got={r[:40]!r:<44} gold={g}")

## 5. Build the training tensors

The part worth understanding: `labels` is set to -100 across every prompt token. -100 is the ignore
index for cross-entropy, so no loss is computed there and the model is graded only on producing the
answer. Skip it and the model also spends capacity learning to predict the customer's message,
which isn't the job.

The answer gets an `eos` appended so the model learns where to stop.

In [ ]:
def encode(ex):
    p_ids = tok(build_prompt(ex["text"], with_labels=False),
                add_special_tokens=False)["input_ids"]
    a_ids = tok(LABELS[ex["label"]] + tok.eos_token,
                add_special_tokens=False)["input_ids"]
    return {"input_ids": (p_ids + a_ids)[:MAXLEN],
            "labels":    ([-100] * len(p_ids) + a_ids)[:MAXLEN]}

train_tok = train_ds.map(encode, remove_columns=train_ds.column_names,
                         desc="tokenizing")

ex = train_tok[0]
print("len", len(ex["input_ids"]), "| supervised tokens:",
      sum(l != -100 for l in ex["labels"]))
print("answer decodes to:",
      tok.decode([l for l in ex["labels"] if l != -100]))

In [ ]:
from dataclasses import dataclass

@dataclass
class PadCollator:
    pad_id: int
    def __call__(self, feats):
        n = max(len(f["input_ids"]) for f in feats)
        out = {"input_ids": [], "labels": [], "attention_mask": []}
        for f in feats:                     # right-pad for training
            d = n - len(f["input_ids"])
            out["input_ids"].append(f["input_ids"] + [self.pad_id] * d)
            out["labels"].append(f["labels"] + [-100] * d)
            out["attention_mask"].append([1] * len(f["input_ids"]) + [0] * d)
        return {k: torch.tensor(v) for k, v in out.items()}

collator = PadCollator(tok.pad_token_id)
_b = collator([train_tok[0], train_tok[1]])
print({k: tuple(v.shape) for k, v in _b.items()})

## 6. LoRA

Rather than update all 500M weights, LoRA freezes them and learns a small low-rank update on each
attention and MLP projection. That leaves under 2% of the parameters trainable, which is what makes
this fit on a free T4.

`r=16` is the rank, and it's the capacity knob. `lora_alpha=32` scales the update; the usual
convention is alpha = 2r. If the model underfits, raise `r` before anything else.

In [ ]:
from peft import LoraConfig, get_peft_model

lora = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()

In [ ]:
from transformers import Trainer, TrainingArguments

model.config.use_cache = False     # incompatible with training; re-enabled after

args = TrainingArguments(
    output_dir="ckpt",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,     # effective batch 16
    learning_rate=LR,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    logging_steps=25,
    save_strategy="no",
    report_to="none",
    fp16=not BF16,
    bf16=BF16,
    seed=SEED,
)

trainer = Trainer(model=model, args=args,
                  train_dataset=train_tok, data_collator=collator)
train_out = trainer.train()

In [ ]:
model.config.use_cache = True
model.eval()
print("final train loss: %.4f" % train_out.training_loss)

## 7. Re-evaluate

Same 300 held-out examples, same scorer. Note `with_labels=False`: the fine-tuned model does this on
a 30 token prompt.

In [ ]:
ft_raw = predict(model, test_texts, with_labels=False)
ft = score(ft_raw, test_gold, "FINE-TUNED (no labels in prompt)")

print("\n" + "=" * 46)
print(f"  accuracy   {base['accuracy']:.1%}  ->  {ft['accuracy']:.1%}"
      f"   ({ft['accuracy'] - base['accuracy']:+.1%})")
print(f"  clean fmt  {base['clean_format']:.1%}  ->  {ft['clean_format']:.1%}")
print("=" * 46)

## 8. Error analysis

Worth more of your attention than the accuracy number. The question is whether the leftover errors
are the model being wrong or the label taxonomy being ambiguous. In Banking77 it's often the latter,
and spotting that is what separates someone who ran a tutorial from someone who thought about the
problem.

In [ ]:
from collections import Counter

conf = Counter((g, p) for g, p in zip(test_gold, ft["preds"]) if g != p)
print(f"{sum(conf.values())} errors over {len(test_gold)} examples\n")
print("most confused pairs (gold -> predicted):")
for (g, p), n in conf.most_common(15):
    print(f"  {n:>2}x  {g:<38} -> {p}")

In [ ]:
# Read the actual failures. Are these really wrong, or is the label pair just ambiguous?
shown = 0
for t, g, p in zip(test_texts, test_gold, ft["preds"]):
    if g != p:
        print(f"text : {t}\ngold : {g}\npred : {p}\n")
        shown += 1
        if shown == 12:
            break

## 9. Save

In [ ]:
import json, os

model.save_pretrained(OUT_DIR)      # adapter only, a few MB
tok.save_pretrained(OUT_DIR)

results = {
    "model": MODEL_ID,
    "dataset": "PolyAI/banking77",
    "n_train": len(train_tok),
    "n_eval": len(test_ds),
    "epochs": EPOCHS,
    "lora": {"r": 16, "alpha": 32, "dropout": 0.05},
    "baseline": {k: base[k] for k in ("accuracy", "clean_format", "unparseable")},
    "finetuned": {k: ft[k] for k in ("accuracy", "clean_format", "unparseable")},
    "final_train_loss": train_out.training_loss,
    "top_confusions": [{"gold": g, "pred": p, "n": n}
                       for (g, p), n in conf.most_common(15)],
}
with open(os.path.join(OUT_DIR, "results.json"), "w") as f:
    json.dump(results, f, indent=2)

print(json.dumps({k: v for k, v in results.items() if k != "top_confusions"}, indent=2))
!du -sh {OUT_DIR}

In [ ]:
# Download the adapter + results, then push the notebook to GitHub.
from google.colab import files
!zip -qr {OUT_DIR}.zip {OUT_DIR}
files.download(f"{OUT_DIR}.zip")

## 10. Results and what to say about them

From this run:

| | baseline | fine-tuned |
|---|---|---|
| accuracy | 25.7% | 92.7% |
| emitted a valid label | 56.0% | 100% |
| unparseable output | 22.3% | 0% |

Final training loss 0.179. The adapter is 45MB. 9,993 training examples, 300 held out, 2 epochs,
roughly 20 minutes on a T4.

**Have you trained or fine-tuned a model yourself?**

LoRA fine-tune of Qwen2.5-0.5B-Instruct on Banking77, a 77-class intent classification set of about
10k examples. Rank-16 adapters on every attention and MLP projection, 8.8M of 503M parameters
trainable (1.75%), 2 epochs on a single T4. Accuracy went from 25.7% zero-shot to 92.7%. The
fine-tuned model no longer needs the 77-label list in its prompt, which cuts the prompt from roughly
700 tokens to 30.

**Hardest problem in that code**

Getting the label pipeline right. `datasets` 5.x dropped support for loading scripts, so the
canonical Banking77 repo stopped loading and I fell back to a parquet mirror. That mirror stores the
label as an integer id with the string in a separate column, so my first attempt produced
`LABELS = [0, 1, 2, ...]`. It passed a length check of 77 and then failed four cells later inside
`"\n".join(LABELS)`. The fix took a minute. The lesson was that the assert should have checked the
type, not the count, and the notebook now does.

Worth mentioning alongside it: `torch.cuda.is_bf16_supported()` returns True on a T4, because it
counts emulated bf16. Real bf16 starts at compute capability 8.0. Trusting that call silently puts
you in a much slower dtype on Turing hardware.

**How do you evaluate and monitor quality in production?**

Here it's a held-out set scored two ways. Accuracy, and separately a format-compliance rate, because
a model that answers correctly in the wrong shape still breaks whatever is calling it. Then a
confusion analysis to see whether the errors trace back to model capacity or to ambiguous label
definitions. In production I'd add prediction logging with confidence, an alert on drift in the
predicted class distribution, and routing of low-confidence cases to a human.

**A recent paper or technique you read or implemented**

LoRA (Hu et al., 2021), implemented here. The claim is that the weight change needed to adapt a
model to a task has low intrinsic rank, so you can represent it as BA with r far smaller than d and
avoid storing a full gradient for every weight.